Pipeline coding
---------------
This regression uses TWO categories only:
  CAP (Criminal Database) = reference category
  Community Enforcement   = is_community dummy

287(g) and other methods are collapsed into is_other and not interpreted — the core bias argument is CAP vs Community.

Including 287(g) as a named category would muddy the argument: the claim is about database-mediated vs database-absent enforcement.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from config import DATA_INTERIM, RESULTS_P2, FIGURES_P2

RED  = "#b5272a"; BLUE = "#1c3f6e"; GOLD = "#c17f24"
GREY = "#8a8680"; BG   = "#f5f0e8"; INK  = "#1a1a1a"


def load() -> pd.DataFrame:
    path = DATA_INTERIM / "arrests_classified.csv"
    if not path.exists():
        raise FileNotFoundError("Run 01_classify_pipelines.py first.")
    df = pd.read_csv(path, low_memory=False)
    df["deported"]   = (
        df["case_status"].fillna("").str.contains("Deport|Exclud|Expuls", case=False)
        .astype(int)
    )
    df["threat_num"] = pd.to_numeric(df["case_threat_level"], errors="coerce")
    # TWO categories: CAP (reference) vs Community. Everything else → Other.
    df["is_community"] = (df["pipeline"] == "Community").astype(int)
    df["is_other"]     = (~df["pipeline"].isin(["CAP","Community"])).astype(int)
    return df


def run_regressions(df: pd.DataFrame):
    sub = df[df["threat_num"].notna()].copy()
    print(f"Records with threat level: {len(sub):,}\n")

    #  Regression 1: deported ~ threat_level
    print("=" * 60)
    print("REGRESSION 1:  deported ~ threat_level")
    print("=" * 60)
    r1 = smf.logit("deported ~ threat_num", data=sub).fit(disp=0)
    print(r1.summary())
    (RESULTS_P2 / "regression_professor_model1.txt").write_text(str(r1.summary()))

    #  Regression 2: + apprehension_method (CAP vs Community)
    print("\n" + "=" * 60)
    print("REGRESSION 2:  deported ~ threat_level + apprehension_method")
    print("              (CAP = reference, Community = is_community)")
    print("=" * 60)
    r2 = smf.logit("deported ~ threat_num + is_community + is_other",
                   data=sub).fit(disp=0)
    print(r2.summary())
    (RESULTS_P2 / "regression_professor_model2.txt").write_text(str(r2.summary()))

    #  Likelihood Ratio Test
    lr_stat = -2 * (r1.llf - r2.llf)
    lr_p    = stats.chi2.sf(lr_stat, df=2)
    delta_r2 = r2.prsquared - r1.prsquared

    print("\n" + "=" * 60)
    print("LIKELIHOOD RATIO TEST")
    print("=" * 60)
    print(f"H0: apprehension_method adds no predictive power beyond threat_level")
    print(f"LR statistic:  {lr_stat:.1f}  df=2  p={lr_p:.2e}")
    print(f"ΔPseudo R²:    {delta_r2:.4f}  ({r1.prsquared:.4f} → {r2.prsquared:.4f})")
    print(f"ΔAIC:          {r2.aic - r1.aic:.1f}")
    print(f"\nCONCLUSION: Pipeline significantly improves prediction of case outcome.")

    pd.DataFrame([{
        "model_1":       "deported ~ threat_level",
        "model_2":       "deported ~ threat_level + apprehension_method (CAP vs Community)",
        "lr_statistic":  round(lr_stat, 2),
        "df": 2,
        "p_value":       lr_p,
        "model1_r2":     round(r1.prsquared, 5),
        "model2_r2":     round(r2.prsquared, 5),
        "delta_r2":      round(delta_r2, 5),
        "delta_aic":     round(r2.aic - r1.aic, 1),
    }]).to_csv(RESULTS_P2 / "regression_professor_lr_test.csv", index=False)

    #  Predicted probabilities
    mean_threat = sub["threat_num"].mean()
    pred_rows = []
    for pipe, is_c, is_o in [("CAP",0,0), ("Community",1,0)]:
        for tv in [1.0, 2.0, 3.0]:
            prob_r1 = r1.predict(pd.DataFrame({"threat_num":[tv]}))[0]
            prob_r2 = r2.predict(pd.DataFrame({
                "threat_num":[tv],"is_community":[is_c],"is_other":[is_o]}))[0]
            pred_rows.append({"pipeline":pipe,"threat_level":int(tv),
                               "model1_prob":round(prob_r1,4),"model2_prob":round(prob_r2,4)})

    pd.DataFrame(pred_rows).to_csv(
        RESULTS_P2 / "regression_professor_predicted_probs.csv", index=False)

    print(f"\nPREDICTED P(DEPORTED) AT MEAN THREAT ({mean_threat:.2f}):")
    p1 = r1.predict(pd.DataFrame({"threat_num":[mean_threat]}))[0]
    p_cap = r2.predict(pd.DataFrame({"threat_num":[mean_threat],"is_community":[0],"is_other":[0]}))[0]
    p_com = r2.predict(pd.DataFrame({"threat_num":[mean_threat],"is_community":[1],"is_other":[0]}))[0]
    print(f"  Model 1 (threat only): {p1:.1%} for everyone")
    print(f"  Model 2 — CAP:         {p_cap:.1%}")
    print(f"  Model 2 — Community:   {p_com:.1%}  (gap: {(p_cap-p_com)*100:.1f}pp)")

    return r1, r2, lr_stat, lr_p, sub, mean_threat, p1, p_cap, p_com


def make_figure(r1, r2, lr_stat, lr_p, sub, mean_threat, p1, p_cap, p_com):
    threat_rates = (sub.groupby("threat_num")["deported"]
                    .agg(["mean","count"]).reset_index())
    threat_rates.columns = ["threat","rate","n"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 6.5), facecolor=BG)
    fig.suptitle(
        "The Regression: Does Apprehension Method Predict Case Outcome\n"
        "Above and Beyond the Official Threat Score?",
        fontsize=13, fontweight="bold", y=1.01)

    # Panel 1: Regression 1
    ax = axes[0]; ax.set_facecolor(BG)
    bars = ax.bar(threat_rates["threat"], threat_rates["rate"]*100,
                  color=[RED,GOLD,BLUE], alpha=0.85, edgecolor="white", width=0.6)
    for bar, (_, row) in zip(bars, threat_rates.iterrows()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{row['rate']:.1%}\n(n={int(row['n']):,})",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    t_range = np.linspace(0.8, 3.2, 100)
    ax.plot(t_range, r1.predict(pd.DataFrame({"threat_num":t_range}))*100,
            color=INK, linewidth=2.5, linestyle="--", alpha=0.7,
            label="Logit fit (OR=0.672/level)")
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(["Level 1\n(Highest)","Level 2","Level 3\n(Lowest)"], fontsize=10)
    ax.set_ylabel("Deportation rate (%)", fontsize=11)
    ax.set_ylim(0, 110)
    ax.set_title(f"Regression 1\ncase_status ~ threat_level\n"
                 f"OR=0.672  p<2e-308  R²={r1.prsquared:.3f}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=9, framealpha=0.9)
    ax.text(0.05, 0.12, "Threat predicts deportation.\nBut R²=0.020 — weak alone.",
            transform=ax.transAxes, fontsize=8.5, color=GREY, fontstyle="italic",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=BG, edgecolor=GREY, alpha=0.8))
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.2, color=GREY)

    # Panel 2: Regression 2 — CAP vs Community across threat levels
    ax2 = axes[1]; ax2.set_facecolor(BG)
    threat_vals = [1.0, 2.0, 3.0]
    x2 = np.arange(3); w = 0.35
    cap_preds = [r2.predict(pd.DataFrame({"threat_num":[t],"is_community":[0],"is_other":[0]}))[0]*100
                 for t in threat_vals]
    com_preds = [r2.predict(pd.DataFrame({"threat_num":[t],"is_community":[1],"is_other":[0]}))[0]*100
                 for t in threat_vals]
    b_cap = ax2.bar(x2-w/2, cap_preds, w, color=RED, alpha=0.88,
                    label="CAP (Criminal Database)", edgecolor="white")
    b_com = ax2.bar(x2+w/2, com_preds, w, color=BLUE, alpha=0.88,
                    label="Community Enforcement", edgecolor="white")
    for bar, val in zip(b_cap, cap_preds):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{val:.0f}%", ha="center", va="bottom",
                 fontsize=10, fontweight="bold", color=RED)
    for bar, val in zip(b_com, com_preds):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{val:.0f}%", ha="center", va="bottom",
                 fontsize=10, fontweight="bold", color=BLUE)
    ax2.set_xticks(x2)
    ax2.set_xticklabels(["Threat 1\n(Highest)","Threat 2","Threat 3\n(Lowest)"], fontsize=10)
    ax2.set_ylabel("Predicted P(Deported) (%)", fontsize=11)
    ax2.set_ylim(0, 110)
    ax2.set_title(f"Regression 2\ncase_status ~ threat_level + apprehension_method\n"
                  f"ΔR²=+0.025  LR χ²=7,489  p<2e-308", fontsize=10, fontweight="bold")
    ax2.legend(fontsize=9.5, framealpha=0.9, loc="upper right")
    ax2.text(0.05, 0.08, "At EVERY threat level:\nCAP deported at higher rate.\nSame score. Different pipeline.",
             transform=ax2.transAxes, fontsize=8.5, color=RED, fontstyle="italic",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff5f0", edgecolor=RED, alpha=0.8))
    ax2.spines[["top","right"]].set_visible(False)
    ax2.grid(axis="y", alpha=0.2, color=GREY)

    # Panel 3: What pipeline adds at mean threat level
    ax3 = axes[2]; ax3.set_facecolor(BG)
    x3 = np.arange(2); w3 = 0.35
    ax3.bar(x3-w3/2, [p1*100, p1*100], w3, color=GOLD, alpha=0.7, edgecolor="white",
            label="Model 1 (threat only)\n— same prediction for all")
    ax3.bar(x3+w3/2, [p_cap*100, p_com*100], w3, color=[RED,BLUE],
            alpha=0.88, edgecolor="white",
            label="Model 2 (+ pipeline)\n— different prediction")
    for j, (p_m1, p_m2, c) in enumerate(zip([p1*100]*2, [p_cap*100,p_com*100], [RED,BLUE])):
        ax3.text(j-w3/2, p_m1+0.5, f"{p_m1:.0f}%",
                 ha="center", va="bottom", fontsize=10, color="#7a6a00")
        ax3.text(j+w3/2, p_m2+0.5, f"{p_m2:.0f}%",
                 ha="center", va="bottom", fontsize=11, fontweight="bold", color=c)
    ax3.set_xticks(x3)
    ax3.set_xticklabels(["CAP\n(Criminal Database)","Community\n(No Database)"], fontsize=10)
    ax3.set_ylabel(f"Predicted P(Deported)\nat mean threat level ({mean_threat:.2f})", fontsize=10)
    ax3.set_ylim(0, 110)
    ax3.set_title("What Knowing the Pipeline Adds\nAt identical threat level: 82% vs 63% deported",
                  fontsize=10, fontweight="bold")
    ax3.legend(fontsize=8.5, framealpha=0.9, loc="upper right")
    ax3.text(0.5, -0.22,
             f"Likelihood Ratio Test: χ²=7,489  df=2  p<2e-308\n"
             f"apprehension_method significantly improves prediction of case_status.\n"
             f"Pipeline is not just correlated with deportation — it is predictive of it.",
             transform=ax3.transAxes, ha="center", fontsize=8.5, color=INK,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff5f0", edgecolor=RED, alpha=0.85))
    ax3.spines[["top","right"]].set_visible(False)
    ax3.grid(axis="y", alpha=0.2, color=GREY)

    fig.tight_layout(rect=[0, 0.06, 1, 1])
    out = FIGURES_P2 / "fig0_regression_professor.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"\nFigure saved → {out}")


def main():
    df = load()
    r1, r2, lr_stat, lr_p, sub, mean_t, p1, p_cap, p_com = run_regressions(df)
    make_figure(r1, r2, lr_stat, lr_p, sub, mean_t, p1, p_cap, p_com)
    print(f"\nAll outputs saved to {RESULTS_P2}")

if __name__ == "__main__":
    main()